# 05 · Detection evaluation: what the pipeline catches

Phase 6, Step 6.5. Every experiment broke a copy of one clean month in one known way
(`configs/faults.yml`); `evaluate.py` compared what the pipeline found with what was broken and
with the unbroken control month. This notebook shows the result per fault and strength, how
much each fault costs the quality score, and what the control month raises by itself.

**Before running:**
```
python src/evaluation/run_experiments.py
```
The last section prints a verdict to paste back.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from evaluation import evaluate as ev
from evaluation import fault_injector as fi

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

## Settings

In [ ]:
config_path = os.environ.get("DATAGUARD_FAULTS_CONFIG")
config = fi.load_faults_config(Path(config_path) if config_path else None)
print(f"target {config.target_month}, history {config.history[0]} to {config.history[-1]}, "
      f"sample {config.sample_fraction:g}, seed {config.seed}")

results = ev.evaluate(config)
ev.store(config, results)
results["strength_label"] = results["experiment_id"].map(
    lambda e: e.split("-", 1)[1] if "-" in e else "-"
)
print(f"{len(results)} experiment(s) evaluated of {len(config.experiments()) - 1} planned")

## 1 · Outcome per fault and strength

**caught**: an incident was raised · **flagged**: the bad rows were quarantined or warned, but
no incident was raised · **missed**: nothing · **error**: the experiment could not run.

In [ ]:
STATUS = {"caught": ("#0ca30c", 3), "flagged": ("#fab219", 2), "missed": ("#d03b3b", 1),
          "error": ("#8a8a85", 0)}
order = [f for f in config.faults if f in set(results["fault_id"])]
def strength_order(label):  # shares of rows first, then days late, then no strength
    if label == "-":
        return (2, 0.0)
    return (1 if label.endswith("d") else 0, float(label.rstrip("pctd")))

labels = sorted(set(results["strength_label"]), key=strength_order)
grid = results.pivot(index="fault_id", columns="strength_label", values="outcome")
grid = grid.reindex(index=order, columns=labels)
grid.fillna("")

In [ ]:
values = grid.map(lambda o: STATUS[o][1] if o in STATUS else None)
scale = [[i / 3, STATUS[name][0]] for name, (_, i) in sorted(STATUS.items(), key=lambda x: x[1][1])]
fig = go.Figure(go.Heatmap(
    z=values.values, x=grid.columns, y=grid.index, text=grid.fillna("").values,
    texttemplate="%{text}",
    colorscale=scale, zmin=0, zmax=3, showscale=False, xgap=2, ygap=2,
    hovertemplate="%{y} at %{x}: %{text}<extra></extra>", hoverongaps=False,
))
fig.update_layout(title="Outcome per fault and strength", yaxis_autorange="reversed",
                  height=60 + 36 * len(grid), plot_bgcolor="#fcfcfb")
fig.show()

In [ ]:
counts = results["outcome"].value_counts().reindex(ev.OUTCOMES, fill_value=0)
counts.to_frame("detectors")

## 2 · Rule faults: broken rows flagged

recall = broken rows the rule flagged; extra_flags = rows flagged beyond the broken rows and
what the control already flags.

In [ ]:
rules = results[results["detector"].str.startswith("rule")]
rules[["experiment_id", "detector", "broken_rows", "caught_rows", "recall", "extra_flags",
       "precision", "incident", "outcome"]]

## 3 · Check faults: what the check saw

The observed and expected values are kept for misses too: they show how far below the check's
limit the fault stayed.

In [ ]:
checks = results[results["detector"].str.startswith("check")]
checks[["experiment_id", "detector", "outcome", "check_severity", "check_observed",
        "check_expected"]]

## 4 · Score impact

Points the month's quality score lost against the control.

In [ ]:
impact = results.dropna(subset=["score_drop"]).sort_values("score_drop")
fig = go.Figure(go.Bar(
    x=impact["score_drop"], y=impact["experiment_id"], orientation="h", marker_color="#3987e5",
    hovertemplate="%{y}: %{x:.1f} points<extra></extra>",
))
fig.update_layout(title="Score drop against the control (points)", height=80 + 18 * len(impact),
                  xaxis_title="points", plot_bgcolor="#fcfcfb", bargap=0.25)
fig.show()

In [ ]:
impact.groupby("fault_id")["score_drop"].agg(["min", "max"]).round(1)

## 5 · Side findings, failed stages and errors

In [ ]:
results[results["side_findings"].notna() | results["stage_failures"].notna()
        | results["error"].notna()][["experiment_id", "side_findings", "stage_failures", "error"]]

## 6 · The control month by itself

Findings here would be false alarms of the setup.

In [ ]:
alarms = ev.control_findings(config)
print("control month raises:", ", ".join(alarms) if alarms else "nothing")

## 7 · Verdict

Copy the printed text back.

In [ ]:
def row_view(r):
    return {"observed": r.check_observed, "expected": r.check_expected} if r.check_observed else {
        "recall": r.recall}

verdict = {
    "experiments": len(results),
    "sample_fraction": config.sample_fraction,
    "outcomes": counts.to_dict(),
    "missed": {r.experiment_id: row_view(r) for r in results.itertuples() if r.outcome == "missed"},
    "flagged_without_incident": sorted(results.loc[results["outcome"] == "flagged", "experiment_id"]),
    "errors": {r.experiment_id: r.error for r in results.itertuples() if r.outcome == "error"},
    "rule_recall_min": None if rules.empty else float(rules["recall"].min()),
    "extra_flags_max": None if rules.empty else int(rules["extra_flags"].max()),
    "control_findings": alarms,
}
print(json.dumps(verdict, indent=2, default=str))